In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --------------------------------------------------
# ÉTAPE 3 — STATISTIQUES DESCRIPTIVES
# Jeu étudié : sales_by_store_date (Silver transformé)
# --------------------------------------------------


spark = (
    SparkSession.builder
    .appName("RetailIntelligence-TP4-Statistiques-Descriptives")
    .master("local[*]").config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

DATA_ROOT = (
    Path.home()
    / "retail-intelligence"
    / "tp3"
    / "data"
)

silver_path = (
    DATA_ROOT
    / "silver-transformed"
    / "sales_by_store_date"
)

print("Silver :", silver_path.exists())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/21 00:41:17 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/21 00:41:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/navid/retail-intelligence/tp4/retail_intel_tp4_venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/21 00:41:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Silver : True


In [2]:
df_silver = spark.read.parquet(str(silver_path))

print("=== 1. STRUCTURE DU JEU DE DONNÉES ===")
df_silver.printSchema()

# Nombre d'enregistrements et de dimensions
nombre_enregistrements = df_silver.count()

print(f"Nombre d'enregistrements : {nombre_enregistrements}")
print(f"Nombre de colonnes : {len(df_silver.columns)}")

# --------------------------------------------------
# 2. STATISTIQUES DES MESURES NUMÉRIQUES
# --------------------------------------------------

mesures = [
    "total_revenue_inc_tax",
    "total_net_paid",
    "total_net_profit",
    "total_quantity_sold",
    "number_of_sales_lines",
    "number_of_tickets"
]

print("\n=== 2. STATISTIQUES DESCRIPTIVES ===")

statistiques = df_silver.select(
    *[
        F.count(F.col(c)).alias(f"{c}_count")
        for c in mesures
    ],
    *[
        F.min(F.col(c)).alias(f"{c}_min")
        for c in mesures
    ],
    *[
        F.max(F.col(c)).alias(f"{c}_max")
        for c in mesures
    ],
    *[
        F.avg(F.col(c)).alias(f"{c}_mean")
        for c in mesures
    ],
    *[
        F.sum(F.col(c)).alias(f"{c}_sum")
        for c in mesures
    ]
)

statistiques.show(truncate=False)

# Médianes : calculées séparément car elles ne sont
# pas additives et peuvent être coûteuses à calculer.
print("\n=== 3. MÉDIANES ===")

for colonne in mesures:
    mediane = df_silver.approxQuantile(colonne, [0.5], 0.01)[0]
    print(f"{colonne} — médiane approximative : {mediane}")

# --------------------------------------------------
# 4. RÉPARTITION PAR MAGASIN
# --------------------------------------------------

print("\n=== 4. RÉPARTITION DES VENTES PAR MAGASIN ===")

repartition_magasin = (
    df_silver.groupBy("store_sk")
    .agg(
        F.count("*").alias("nombre_jours_magasin"),
        F.sum("total_revenue_inc_tax").alias("ca_ttc"),
        F.sum("total_quantity_sold").alias("quantites_vendues"),
        F.sum("number_of_tickets").alias("nombre_tickets")
    )
    .orderBy(F.desc("ca_ttc"))
)

repartition_magasin.show(50, truncate=False)

# --------------------------------------------------
# 5. RÉPARTITION PAR DATE
# --------------------------------------------------
# Cette sortie présente les résultats par date_sk.
# Elle ne constitue une évolution chronologique
# que si date_sk est interprétable ou reliée à une
# dimension calendrier.

print("\n=== 5. RÉPARTITION PAR DATE (date_sk) ===")

repartition_date = (
    df_silver.groupBy("date_sk")
    .agg(
        F.sum("total_revenue_inc_tax").alias("ca_ttc"),
        F.sum("total_quantity_sold").alias("quantites_vendues"),
        F.sum("number_of_tickets").alias("nombre_tickets")
    )
    .orderBy("date_sk")
)

repartition_date.show(50, truncate=False)

# --------------------------------------------------
# 6. CONTRÔLES DE COHÉRENCE
# --------------------------------------------------

print("\n=== 6. CONTRÔLES DE COHÉRENCE ===")

print("Nombre de magasins distincts :",
      df_silver.select("store_sk").distinct().count())

print("Nombre de dates distinctes :",
      df_silver.select("date_sk").distinct().count())

print("Nombre de couples magasin-date distincts :",
      df_silver.select("store_sk", "date_sk").distinct().count())

print("Nombre de lignes avec une clé magasin ou date NULL :",
      df_silver.filter(
          F.col("store_sk").isNull() |
          F.col("date_sk").isNull()
      ).count())

=== 1. STRUCTURE DU JEU DE DONNÉES ===
root
 |-- store_sk: long (nullable = true)
 |-- date_sk: long (nullable = true)
 |-- total_revenue_inc_tax: decimal(17,2) (nullable = true)
 |-- total_net_paid: decimal(17,2) (nullable = true)
 |-- total_net_profit: decimal(17,2) (nullable = true)
 |-- total_quantity_sold: long (nullable = true)
 |-- number_of_sales_lines: long (nullable = true)
 |-- number_of_tickets: long (nullable = true)
 |-- _grain: string (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_original_file: string (nullable = true)
 |-- _processing_stage: string (nullable = true)

Nombre d'enregistrements : 21430
Nombre de colonnes : 12

=== 2. STATISTIQUES DESCRIPTIVES ===


26/11/21 00:41:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------------------------+--------------------+----------------------+-------------------------+---------------------------+-----------------------+-------------------------+------------------+--------------------+-----------------------+-------------------------+---------------------+-------------------------+------------------+--------------------+-----------------------+-------------------------+---------------------+--------------------------+-------------------+---------------------+------------------------+--------------------------+----------------------+-------------------------+------------------+--------------------+-----------------------+-------------------------+---------------------+
|total_revenue_inc_tax_count|total_net_paid_count|total_net_profit_count|total_quantity_sold_count|number_of_sales_lines_count|number_of_tickets_count|total_revenue_inc_tax_min|total_net_paid_min|total_net_profit_min|total_quantity_sold_min|number_of_sales_lines_min|number_of_tickets_min|t

total_revenue_inc_tax — médiane approximative : 51479.12
total_net_paid — médiane approximative : 49246.2
total_net_profit — médiane approximative : -23964.67
total_quantity_sold — médiane approximative : 1466.0
number_of_sales_lines — médiane approximative : 29.0
number_of_tickets — médiane approximative : 4.0

=== 4. RÉPARTITION DES VENTES PAR MAGASIN ===
+--------+--------------------+------------+-----------------+--------------+
|store_sk|nombre_jours_magasin|ca_ttc      |quantites_vendues|nombre_tickets|
+--------+--------------------+------------+-----------------+--------------+
|7       |1792                |103384198.80|2915053          |7645          |
|4       |1791                |101130648.13|2832572          |7462          |
|9       |1784                |101109360.86|2830684          |7435          |
|0       |1786                |101047840.24|2830581          |7456          |
|8       |1789                |100916146.68|2821869          |7414          |
|3       |1786  

In [3]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.
